In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

tr = pd.read_parquet('data/train.parquet')
te = pd.read_parquet('data/test.parquet')
courses = pd.read_parquet('data/courses.parquet')
ass = pd.read_parquet('data/assessments.parquet')
sa = pd.read_parquet('data/student_assessment.parquet')
vle = pd.read_parquet('data/vle.parquet')
sv = pd.read_parquet('data/student_vle.parquet')
keys = ['code_module', 'code_presentation']
cats = tr.select_dtypes(include=['object', 'str']).columns.tolist()


In [ ]:
g = sv.groupby('enrollment_id')
act = pd.DataFrame()
act['active_days'] = g.date.nunique()
act['last_activity'] = g.date.max()
act['unique_sites'] = g.id_site.nunique()

sites = vle[vle.activity_type.isin(['page', 'oucontent', 'dualpane'])]
clicks = sv.merge(sites[['id_site', 'activity_type']], on='id_site')
clicks = clicks.pivot_table(index='enrollment_id', columns='activity_type', values='sum_click', aggfunc='sum', fill_value=0)
clicks = clicks.add_prefix('click_')

late = sv[sv.date.between(40, 59)]
windows = late.copy()
windows['window'] = windows.date // 10
g = windows.groupby(['enrollment_id', 'window'])
days = g.date.nunique()
days = days.unstack(fill_value=0)
days = days.rename(columns={4: 'days_4', 5: 'days_5'})
act = act.join(clicks)
act = act.join(days)

g = sv.groupby('enrollment_id')
all_clicks = g.sum_click.sum()
g = late.groupby('enrollment_id')
late_clicks = g.sum_click.sum()

is_content = vle.activity_type.eq('oucontent')
is_used = vle.id_site.isin(sv.id_site.unique())
sites = vle[is_content & is_used]
g = sites.groupby(keys)
site_count = g.id_site.nunique()
site_count = site_count.reset_index(name='n_sites')


In [ ]:
cols = ['id_assessment', 'date', 'weight'] + keys
s = sa.merge(ass[cols], on='id_assessment', how='left')
s['delay'] = s.date_submitted - s.date

g = s.groupby('enrollment_id')
grades = pd.DataFrame()
grades['assessments_done'] = g.id_assessment.nunique()
grades['score_mean'] = g.score.mean()
grades['score_min'] = g.score.min()
grades['score_max'] = g.score.max()
grades['last_submission'] = g.date_submitted.max()
grades['mean_delay'] = g.delay.mean()

early_ass = ass[ass.date.le(60)]
g = early_ass.groupby(keys)
due = pd.DataFrame()
due['assessments_due'] = g.id_assessment.count()
due['weight_due'] = g.weight.sum()
due = due.reset_index()

cols = ['score', 'date_submitted']
g = s.groupby('id_assessment')[cols]
ranks = g.rank(pct=True)
count = g.transform('count')
course_ranks = s.groupby(keys)[cols].rank(pct=True)
ranks = ranks.where(count >= 30, course_ranks)
s[['score_percentile', 'time_pct']] = ranks
s = s.sort_values(['enrollment_id', 'date_submitted', 'id_assessment'])
g = s.groupby('enrollment_id').score_percentile
p = g.agg(['first', 'last'])
p['change'] = p['last'] - p['first']

def weighted_mean(df, col):
    g = df.groupby('enrollment_id')
    values = df[col] * df.weight
    total = values.groupby(df.enrollment_id).sum()
    weight = g.weight.sum()
    weight = weight.replace(0, np.nan)
    mean = total / weight
    fallback = g[col].mean()
    return mean.fillna(fallback)

valid_scores = s.dropna(subset=['score_percentile'])
weighted = weighted_mean(valid_scores, 'score_percentile')
time_pct = weighted_mean(s, 'time_pct')

early = s[s.date.le(60)].copy()
score = early.score.fillna(0)
early['earned_weight'] = early.weight * score / 100
on_time = early.date_submitted.le(early.date)
early['ontime_weight'] = early.weight.where(on_time, 0)
g = early.groupby('enrollment_id')
weights = g[['weight', 'earned_weight', 'ontime_weight']].sum()


In [ ]:
df = pd.concat([tr.drop(columns='target'), te], ignore_index=True)
df = df.merge(courses, on=keys, how='left')
df = df.merge(due, on=keys, how='left')
df = df.join(grades, on='enrollment_id')
df = df.join(act, on='enrollment_id')
df = df.merge(site_count, on=keys, how='left')
zero = ['assessments_due', 'weight_due', 'assessments_done', 'active_days', 'unique_sites', 'click_page', 'click_oucontent', 'click_dualpane', 'days_4', 'days_5']
df[zero] = df[zero].fillna(0)
df[cats] = df[cats].fillna('missing')
ids = df.enrollment_id
limit = df.weight_due.clip(lower=1)

df['total_clicks'] = ids.map(all_clicks)
df['total_clicks'] = df.total_clicks.fillna(0)
active_days = df.active_days.clip(lower=1)
df['clicks_per_active_day'] = df.total_clicks / active_days
late_total = ids.map(late_clicks)
late_total = late_total.fillna(0)
click_total = df.total_clicks.clip(lower=1)
df['late_click_share'] = late_total / click_total
n_sites = df.n_sites.fillna(0)
n_sites = n_sites.clip(lower=1)
df['click_oucontent'] = df.click_oucontent / n_sites
df = df.rename(columns={'click_oucontent': 'oucontent_per_site'})

df['score_last_percentile'] = ids.map(p['last'])
df['score_change_percentile'] = ids.map(p['change'])
completed = ids.map(weights.weight)
completed = completed.fillna(0)
df['completed_due_weight_share'] = completed / limit
g = df.groupby(keys)
df['unique_sites_percentile'] = g.unique_sites.rank(pct=True)
earned = ids.map(weights.earned_weight)
earned = earned.fillna(0)
df['earned_due_weight_share'] = earned / limit
ontime = ids.map(weights.ontime_weight)
ontime = ontime.fillna(0)
df['ontime_due_weight_share'] = ontime / limit
df['weighted_score_percentile'] = ids.map(weighted)
x = df.drop(columns=['enrollment_id', 'student_id', 'unique_sites', 'n_sites'])

num = ['score_mean', 'last_activity', 'days_5', 'total_clicks']
scaled = x[num].copy()
for _, rows in x.groupby(keys).groups.items():
    part = x.loc[rows, num]
    scaler = RobustScaler()
    scaled.loc[rows, num] = scaler.fit_transform(part)
robust = scaled.add_suffix('_robust')
x = x.join(robust)
x2 = x.copy()
x2['submission_time_pct'] = ids.map(time_pct)


In [ ]:
n = len(tr)
y = tr.target
cb = CatBoostClassifier(
    depth=6, iterations=400, learning_rate=0.03, l2_leaf_reg=10,
    random_seed=42, verbose=False, allow_writing_files=False, thread_count=4)
cb.fit(x.iloc[:n], y, cat_features=cats)
p_base = cb.predict_proba(x.iloc[n:])[:, 1]

cx = x.drop(columns=keys)
course_cats = [c for c in cats if c not in keys]
p_course = p_base.copy()
xa = cx.iloc[:n]
xb = cx.iloc[n:]
for course in ['BBB', 'DDD', 'FFF']:
    a = tr.code_module.eq(course).to_numpy()
    b = te.code_module.eq(course).to_numpy()
    model = clone(cb)
    model.set_params(depth=5)
    train_x = xa.loc[a]
    test_x = xb.loc[b]
    model.fit(train_x, y[a], cat_features=course_cats)
    p_course[b] = model.predict_proba(test_x)[:, 1]


In [ ]:
xl = x2.drop(columns='code_presentation').copy()
xl['season'] = x.code_presentation.str[-1]
year = x.code_presentation.str[:4]
year = year.astype(int)
xl['year'] = year - 2013
cat_lgb = [c for c in cats if c != 'code_presentation']
cat_lgb.append('season')
for c in cat_lgb:
    xl[c] = xl[c].astype('category')
lgb = LGBMClassifier(num_leaves=15, max_depth=6, min_child_samples=60, n_estimators=500,
    learning_rate=0.03, reg_lambda=10, extra_trees=True, path_smooth=20,
    n_jobs=4, verbosity=-1)

p_cb, p_lgb = [], []
for seed in [42, 2026]:
    model = clone(cb)
    model.set_params(random_seed=seed)
    model.fit(x2.iloc[:n], y, cat_features=cats)
    p_cb.append(model.predict_proba(x2.iloc[n:])[:, 1])
    model = clone(lgb)
    model.set_params(random_state=seed)
    model.fit(xl.iloc[:n], y)
    p_lgb.append(model.predict_proba(xl.iloc[n:])[:, 1])

mean_lgb = np.mean(p_lgb, axis=0)
mean_cb = np.mean(p_cb, axis=0)
boost = 0.34 * p_base + 0.16 * p_course
boost = boost + 0.30 * mean_lgb
boost = boost + 0.20 * mean_cb


In [ ]:
def daily(values):
    days = values.unstack()
    days = days.reindex(index=ids, columns=range(-25, 60))
    days = days.fillna(0)
    return days.to_numpy()

clicks = sv[sv.date.between(-25, 59)]
g = clicks.groupby(['enrollment_id', 'date'])
clicks = daily(g.sum_click.sum())
submissions = sa[sa.date_submitted.between(-25, 59)]
g = submissions.groupby(['enrollment_id', 'date_submitted'])

click_log = np.log1p(clicks)
click_log = click_log.clip(0, 6) / 6
active = clicks > 0
score = daily(g.score.mean())
score = score / 100
count = daily(g.size())
count = np.log1p(count)
count = count.clip(0, 3) / 3
seq = np.stack([click_log, active, score, count], axis=-1)
seq = seq.astype('float32')


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(2)

tab = x2.drop(columns=['code_module', 'code_presentation', 'module_presentation_length', 'assessments_due', 'weight_due'])
cat = [c for c in cats if c in tab]
num = [c for c in tab if c not in cat]
num_prep = make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler())
cat_prep = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
prep = ColumnTransformer([('num', num_prep, num), ('cat', cat_prep, cat)])
prep.fit(tab.iloc[:n])

values = prep.transform(tab)
values = values.clip(-8, 8)
values = np.array(values, dtype='float32', order='C')
values = torch.from_numpy(values)
seq = torch.from_numpy(seq)
target = torch.tensor(y.to_numpy(), dtype=torch.float32)
train = TensorDataset(values[:n], seq[:n], target)
test = TensorDataset(values[n:], seq[n:])


In [ ]:
class StudentModel(nn.Module):
    def __init__(self, n_features, kind):
        super().__init__()
        self.with_tab = kind == 'LSTM'
        rnn = nn.LSTM if self.with_tab else nn.RNN
        self.history = rnn(input_size=4, hidden_size=32, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(n_features + 64 if self.with_tab else 64, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, tab, days):
        h, _ = self.history(days)

        parts = [h[:, -1], h.mean(dim=1)]
        if self.with_tab:
            parts = [tab] + parts
        x = torch.cat(parts, dim=1)
        x = self.head(x)
        return x.flatten()


In [ ]:
def fit_model(kind, seed, epochs):
    torch.manual_seed(seed)
    net = StudentModel(values.shape[1], kind)
    opt = torch.optim.AdamW(net.parameters(), lr=0.001, weight_decay=0.01)
    loss = nn.BCEWithLogitsLoss()
    loader = DataLoader(train, batch_size=256, shuffle=True)

    net.train()
    for _ in range(epochs):
        for tab, days, target in loader:
            opt.zero_grad()
            err = loss(net(tab, days), target)
            err.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()

    net.eval()
    pred = []
    with torch.no_grad():
        for tab, days in DataLoader(test, batch_size=512):
            p = net(tab, days).sigmoid()
            pred.append(p.numpy())
    pred = np.concatenate(pred)
    print(f'{kind} seed={seed} yay')
    return pred


p_lstm = []
for seed in [42, 2026]:
    p_seed = fit_model('LSTM', seed, 8)
    p_lstm.append(p_seed)
lstm = np.mean(p_lstm, axis=0)

p_rnn = []
for seed in [42, 2026]:
    p_seed = fit_model('RNN', seed, 15)
    p_rnn.append(p_seed)
rnn = np.mean(p_rnn, axis=0)


In [ ]:
pred = 0.825 * boost + 0.10 * lstm + 0.075 * rnn
submission = pd.DataFrame({'id': te.enrollment_id, 'prediction': pred})
assert len(submission) == len(te) and submission.id.is_unique
assert np.isfinite(pred).all() and submission.prediction.between(0, 1).all()
submission.to_csv('submission.csv', index=False)
submission.head()


In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut

flow = tr.code_presentation
scores = {}
for a, b in LeaveOneGroupOut().split(tr, y, groups=flow):
    model = clone(cb)
    model.fit(x.iloc[a], y.iloc[a], cat_features=cats)
    pred_cv = model.predict_proba(x.iloc[b])[:, 1]
    scores[flow.iloc[b[0]]] = roc_auc_score(y.iloc[b], pred_cv)

cv = pd.Series(scores, name='ROC AUC').to_frame()
print(f"Общая CV: {cv['ROC AUC'].mean():.6f}")
cv.round(6)
